<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Neurons and layers

**[Neurons and layers](https://learning.nextia-ai.com/courses/deep-learning/m01/neurons-and-layers/)** · Deep Learning and Transformers Explained · Module 1, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute one neuron by hand, see that it is the logistic regression of the machine learning course, compute a layer of neurons, and count the parameters of a network.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | No account and no data download. The first cell checks PyTorch; Colab and Kaggle have it. |
| **You should know** | Logistic regression, from [Classification intuition](https://learning.nextia-ai.com/courses/ml/m04/classification-intuition/) in the machine learning course. |
| **You do not need** | Any earlier notebook of this course. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m01/neurons-and-layers/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M01-L02-neurons-and-layers/neurons-and-layers-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One neuron by hand

A neuron computes a **weighted sum** of its inputs, adds a **bias**, and then applies an **activation**. Here the activation is the logistic function, the S-curve of the machine learning course.

The weights and the bias below are the escalation model of [Classification intuition](https://learning.nextia-ai.com/courses/ml/m04/classification-intuition/#several-features), rounded to four decimals. The ticket has priority 1, 250 words, 3 earlier tickets and was written at 10 in the morning.

> **Predict.** Which input adds the most to `z`: priority, with the largest weight, or another one? Write your prediction, then run the cell.

In [ ]:
import math

bias = -0.7279
weights = {"priority": -0.8938, "word_count": 0.0052, "prior_tickets_90d": 0.0233, "created_hour": -0.0096}
ticket = {"priority": 1, "word_count": 250, "prior_tickets_90d": 3, "created_hour": 10}

def logistic(z):
    return 1 / (1 + math.exp(-z))

z = bias
for name in weights:
    product = weights[name] * ticket[name]
    print(f"{name:18} {ticket[name]:>4} x {weights[name]:+.4f} = {product:+.4f}")
    z = z + product
print(f"z = {z:.4f}, score = {logistic(z):.4f}")

> **Check.** You should see `z = -0.3478, score = 0.4139`. The largest term is `word_count`, +1.3000, although its weight looks tiny: the effect of an input is weight × input.

This is the neuron of the lesson's diagram, and the same computation as logistic regression's `predict_proba`.

## 2. Your turn: a neuron with two inputs

> **Your turn.** A neuron has the weights 0.5 and −1 and the bias 0.2. Its inputs are 3 and 1. Compute its weighted sum plus bias in `z_practice`, and its output with the logistic function in `out_practice`. Then run the check cells.

In [ ]:
z_practice = ...    # the weighted sum plus the bias
out_practice = ...  # logistic(z_practice)

In [ ]:
expect('z_practice', z_practice, 0.7)

In [ ]:
expect('out_practice', out_practice, 0.6682, tolerance=0.001)

## 3. A neuron is a logistic regression

Now check the claim of the lesson with scikit-learn. The next cell makes 200 toy points with two features (scikit-learn's `make_moons`), fits a `LogisticRegression`, and reads its learned `coef_` (the weights) and `intercept_` (the bias). Then it computes the first three points **by hand**, as one neuron, and compares with `predict_proba`.

> **Predict.** Will the hand-computed scores and scikit-learn's scores agree to 4 decimals?

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression

X, y = make_moons(200, noise=0.25, random_state=0)
model = LogisticRegression().fit(X, y)
w1, w2 = model.coef_[0]
b = model.intercept_[0]
print(f"weights {w1:.4f} and {w2:.4f}, bias {b:.4f}")
for x1, x2 in X[:3]:
    by_hand = logistic(w1 * x1 + w2 * x2 + b)
    sklearn_score = model.predict_proba([[x1, x2]])[0, 1]
    print(f"point ({x1:+.3f}, {x2:+.3f}): by hand {by_hand:.4f}, scikit-learn {sklearn_score:.4f}")

> **Check.** For each point, the two scores are the same. The exact weights can differ a little between scikit-learn versions, but the two columns always agree.

So one neuron with the logistic activation **is** a logistic regression: the same weighted sum, the same bias, the same S-curve.

## 4. A layer: three neurons reading the same inputs

A **layer** is several neurons that read the same inputs, each with its own weights and bias. The next cell computes the lesson's tiny layer for the inputs 2 and 1, first with a loop, then with NumPy in one line. `W @ x` multiplies each row of weights with the inputs and adds them up; Module 2 explains this matrix product.

Run the cell.

In [ ]:
x = [2.0, 1.0]
layer = [  # (weights, bias) of each neuron
    ([1.0, 1.0], -1.0),
    ([1.0, -1.0], 0.0),
    ([-0.5, 2.0], 0.5),
]
outputs = []
for w, b in layer:
    outputs.append(w[0] * x[0] + w[1] * x[1] + b)
print("with a loop: ", outputs)

W = np.array([w for w, _ in layer])
bias_vector = np.array([b for _, b in layer])
print("with NumPy:  ", W @ np.array(x) + bias_vector)

> **Check.** Both lines show the three outputs 2, 1 and 1.5. Two numbers went in; three came out, one per neuron.

## 5. Count the parameters

A layer with `n` inputs and `m` neurons has n × m weights and m biases. A network is layers in a chain, so its parameters are the sum over its layers.

> **Your turn.** Write the function `count_parameters(sizes)`. `sizes` lists the layer sizes, from the inputs to the outputs, for example `[2, 4, 1]`. For each pair of neighbouring sizes `n` and `m`, add n × m + m. Return the total. Then run the check cell: it tries three networks.

In [ ]:
def count_parameters(sizes):
    total = 0
    ...  # for each pair (n, m) of neighbouring sizes, add n * m + m
    return total

In [ ]:
expect_hash('your counts for [2, 4, 1], [4, 3, 2] and [2864, 64, 5]', [count_parameters([2, 4, 1]), count_parameters([4, 3, 2]), count_parameters([2864, 64, 5])], '1ba531a3b0f87fb0')

The ticket network of this course, 2,864 word counts → 64 hidden neurons → 5 team scores, has 183,685 parameters. PyTorch can count them too. You do not need to understand this code yet; Module 2 introduces PyTorch. `nn.Linear(2864, 64)` is one layer with 2,864 inputs and 64 neurons, and `nn.ReLU()` is the activation of the next lesson.

Run the cell.

In [ ]:
from torch import nn

ticket_network = nn.Sequential(nn.Linear(2864, 64), nn.ReLU(), nn.Linear(64, 5))
print("PyTorch counts", sum(p.numel() for p in ticket_network.parameters()), "parameters")

> **Check.** You should see `PyTorch counts 183685 parameters`, the same as your function.

## 6. Change one thing: 128 hidden neurons

> **Predict.** Tomás wants 128 hidden neurons instead of 64. Does the number of parameters double, more than double, or less than double? Write your prediction, then set `wider` to the parameter count of `[2864, 128, 5]` and run the check.

In [ ]:
wider = ...  # the parameters of 2864 -> 128 -> 5

In [ ]:
expect('wider', wider, 367365)

It almost exactly doubles: nearly every parameter sits between the 2,864 inputs and the hidden layer, and that part grows in step with the hidden neurons.

## 7. Failure case: a small weight read as a small effect

Tomás says: "word_count has the smallest weight, so it hardly matters." Section 1 already showed the opposite. The next cell sorts the four inputs by the size of the weight, and then by the size of weight × input.

Run the cell.

In [ ]:
by_weight = sorted(weights, key=lambda k: abs(weights[k]), reverse=True)
by_effect = sorted(weights, key=lambda k: abs(weights[k] * ticket[k]), reverse=True)
print("largest weight first:", by_weight)
print("largest effect first:", by_effect)

> **Check.** By weight, `word_count` is last; by effect, it is first.

A weight is "per unit of the input". To see what drives one output, look at weight × input. To compare weights with each other, put the inputs on one scale first.

## Next

You can now compute a neuron, see that it is a logistic regression, compute a layer and count a network's parameters. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m01/neurons-and-layers/) has the diagrams and the knowledge checks that count toward your certificate. The next lesson shows what happens when the layers have no activation, and why the activation is essential.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Why layers need activations](https://learning.nextia-ai.com/courses/deep-learning/m01/activations/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m01/neurons-and-layers/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The toy data is generated by scikit-learn and has no real people.